# 06 — Scenario Analysis

**Project:** Repeat Purchase Recovery — Olist E-commerce (Brazil)  
**Objective:** Quantify business impact under different uplift assumptions  
**Status:** What-if estimates based on validated findings, not forecasts

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

DB_PATH = 'data/olist.db'
IMG_PATH = 'images'
conn = sqlite3.connect(DB_PATH)
os.makedirs(IMG_PATH, exist_ok=True)

C_BLUE = '#2563EB'
C_ORANGE = '#EA580C'
C_GREY = '#94A3B8'
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print('Connected to:', DB_PATH)

# ═══════════════════════════════════════════════════════════════════
# VALIDATED REVENUE BENCHMARKS FROM NB05
# ═══════════════════════════════════════════════════════════════════
# These are derived from repeat_vs_one_time_summary analysis
REV_REPEAT_BUYER_90D = 253.10      # Repeat buyer avg 90-day revenue
REV_ONETIME_BUYER_90D = 160.76     # One-time buyer avg 90-day revenue
REV_REPEAT_UPLIFT_90D = REV_REPEAT_BUYER_90D - REV_ONETIME_BUYER_90D  # ~R$92 incremental
REV_LATE_DELIVERY_LTV = 178.10     # Late-delivery customers lifetime revenue

print(f'\n📊 VALIDATED BENCHMARKS (from NB05):')
print(f'  Repeat buyer 90-day revenue:     R${REV_REPEAT_BUYER_90D:.2f}')
print(f'  One-time buyer 90-day revenue:   R${REV_ONETIME_BUYER_90D:.2f}')
print(f'  Incremental uplift:              R${REV_REPEAT_UPLIFT_90D:.2f}')
print(f'  Late-delivery lifetime revenue:  R${REV_LATE_DELIVERY_LTV:.2f}')

## 1. Baseline Metrics

In [ ]:
baseline = pd.read_sql('SELECT total_customers, repeat_buyers, ROUND(raw_repeat_purchase_rate*100,2) AS repeat_pct FROM executive_kpi_summary', conn).iloc[0]
total_customers = baseline['total_customers']
baseline_repeat_buyers = baseline['repeat_buyers']
baseline_repeat_rate = baseline['repeat_pct']

print(f'\n═══════════════════════════════════════════════════════════════════')
print(f'BASELINE (Observed): 23-month window (Sept 2016 – Aug 2018)')
print(f'═══════════════════════════════════════════════════════════════════')
print(f'Total customers:             {int(total_customers):,}')
print(f'Repeat buyers (lifetime):    {int(baseline_repeat_buyers):,}')
print(f'Raw repeat rate:             {baseline_repeat_rate:.2f}%')
print(f'\nNote: All scenarios below show 90-day revenue (not annualized)')

## 2. Scenario 1: Cross-Sell (Medium-Case: +1.0pp)

In [ ]:
basket = pd.read_sql('SELECT items_in_first_order, COUNT(*) AS customers, ROUND(AVG(CAST(r.repeat_within_90d AS REAL))*100,2) AS repeat_rate_90d FROM csm_mat c JOIN crsm_mat r ON c.customer_unique_id = r.customer_unique_id WHERE items_in_first_order IN (1,2,3) AND r.repeat_within_90d IS NOT NULL GROUP BY items_in_first_order ORDER BY items_in_first_order', conn)

single = basket[basket['items_in_first_order']==1].iloc[0]
multi_2 = basket[basket['items_in_first_order']==2].iloc[0]

s1_customers = single['customers']
s1_rate = single['repeat_rate_90d'] / 100
multi_rate = multi_2['repeat_rate_90d'] / 100

print(f'\n═══════════════════════════════════════════════════════════════════')
print(f'SCENARIO 1: CROSS-SELL PROGRAM (Multi-Item Basket Conversion)')
print(f'═══════════════════════════════════════════════════════════════════')
print(f'Target: Single-item first order buyers → multi-item buyers')
print(f'')
print(f'Customers with single-item first order:  {int(s1_customers):,}')
print(f'  Current 90-day repeat rate:            {s1_rate*100:.2f}%')
print(f'  Current repeaters:                     {int(s1_customers * s1_rate):,}')
print(f'')
print(f'Repeat rate for 2-item buyers:           {multi_rate*100:.2f}%')
print(f'  Observed repeat rate uplift:           +{(multi_rate - s1_rate)*100:.2f}pp')
print(f'')
print(f'REVENUE FORMULA (Scenario 1):')
print(f'  incremental_revenue = incremental_buyers × R${REV_REPEAT_UPLIFT_90D:.2f}')
print(f'  (using 90-day revenue uplift benchmark from NB05)')
print(f'')

results_s1 = []
for uplift_pp in [0.5, 1.0, 2.0]:
    uplift_pct = uplift_pp / 100
    new_rate = s1_rate + uplift_pct
    incremental = int(s1_customers * (new_rate - s1_rate))
    revenue = incremental * REV_REPEAT_UPLIFT_90D
    
    results_s1.append({
        'uplift_pp': uplift_pp,
        'incremental_buyers': incremental,
        'revenue_90d_m': revenue / 1_000_000
    })
    
    print(f'Uplift +{uplift_pp:.1f}pp scenario:')
    print(f'  New repeat rate:                       {new_rate*100:.2f}%')
    print(f'  Incremental repeat buyers:             {incremental:,}')
    print(f'  Incremental revenue (90-day):          R${revenue/1_000_000:.2f}M')
    print(f'')

s1_df = pd.DataFrame(results_s1)

In [ ]:
sanity_s1 = s1_df.copy()
sanity_s1['revenue_per_buyer'] = sanity_s1['revenue_90d_m'] * 1_000_000 / sanity_s1['incremental_buyers']
sanity_s1['benchmark'] = REV_REPEAT_UPLIFT_90D
sanity_s1['within_tolerance'] = np.abs(sanity_s1['revenue_per_buyer'] - sanity_s1['benchmark']) < 1

print('SANITY CHECK TABLE (Scenario 1):')
print(f'{"Scenario":<15} {"Incr. Buyers":>15} {"Total Revenue":>15} {"Rev/Buyer":>15} {"Benchmark":>15} {"✓":>4}')
print('─' * 70)
for _, row in sanity_s1.iterrows():
    scenario = f"+{row['uplift_pp']:.1f}pp"
    buyers = f"{int(row['incremental_buyers']):,}"
    total = f"R${row['revenue_90d_m']:.2f}M"
    per_buyer = f"R${row['revenue_per_buyer']:.0f}"
    bench = f"R${row['benchmark']:.0f}"
    status = '✓' if row['within_tolerance'] else '✗'
    print(f"{scenario:<15} {buyers:>15} {total:>15} {per_buyer:>15} {bench:>15} {status:>4}")
print()

In [ ]:
delay = pd.read_sql('''
SELECT delivery_status, COUNT(*) AS customers, 
       ROUND(AVG(CAST(r.repeat_within_90d AS REAL))*100,2) AS repeat_rate_90d
FROM csm_mat c
JOIN crsm_mat r ON c.customer_unique_id = r.customer_unique_id
WHERE delivery_status IS NOT NULL AND r.repeat_within_90d IS NOT NULL
GROUP BY delivery_status
''', conn)

late = delay[delay['delivery_status']=='late'].iloc[0]
early = delay[delay['delivery_status']=='early'].iloc[0]

late_customers = late['customers']
late_rate = late['repeat_rate_90d'] / 100
early_rate = early['repeat_rate_90d'] / 100
gap_pp = (early_rate - late_rate) * 100

print(f'\n═══════════════════════════════════════════════════════════════════')
print(f'SCENARIO 2: DELIVERY-DELAY RECOVERY TOUCHPOINT')
print(f'═══════════════════════════════════════════════════════════════════')
print(f'Target: Late-delivery customers (service recovery)')
print(f'')
print(f'Late-delivery customers:                 {int(late_customers):,}')
print(f'  Current 90-day repeat rate:            {late_rate*100:.2f}%')
print(f'  Early-delivery repeat rate:            {early_rate*100:.2f}%')
print(f'  Repeat rate gap:                       {gap_pp:.2f}pp')
print(f'')
print(f'REVENUE FORMULA (Scenario 2):')
print(f'  incremental_revenue = incremental_buyers × R${REV_LATE_DELIVERY_LTV:.2f}')
print(f'  (using late-delivery avg lifetime revenue from NB05)')
print(f'  Note: This is lifetime value, not 90-day')
print(f'')

results_s2 = []
for recovery in [0.25, 0.50, 0.75]:
    gap_recovery_pp = gap_pp * recovery
    new_rate = late_rate + (gap_recovery_pp / 100)
    incremental = int(late_customers * (new_rate - late_rate))
    revenue = incremental * REV_LATE_DELIVERY_LTV
    
    results_s2.append({
        'recovery_pct': recovery * 100,
        'incremental_buyers': incremental,
        'revenue_ltv_m': revenue / 1_000_000
    })
    
    print(f'Recovery {int(recovery*100)}% of gap ({gap_recovery_pp:.2f}pp):')
    print(f'  New repeat rate:                       {new_rate*100:.2f}%')
    print(f'  Incremental repeat buyers:             {incremental:,}')
    print(f'  Incremental revenue (lifetime):        R${revenue/1_000_000:.3f}M')
    print(f'')

s2_df = pd.DataFrame(results_s2)

In [ ]:
sanity_s2 = s2_df.copy()
sanity_s2['revenue_per_buyer'] = sanity_s2['revenue_ltv_m'] * 1_000_000 / sanity_s2['incremental_buyers']
sanity_s2['benchmark'] = REV_LATE_DELIVERY_LTV
sanity_s2['within_tolerance'] = np.abs(sanity_s2['revenue_per_buyer'] - sanity_s2['benchmark']) < 1

print('SANITY CHECK TABLE (Scenario 2):')
print(f'{"Scenario":<15} {"Incr. Buyers":>15} {"Total Revenue":>15} {"Rev/Buyer":>15} {"Benchmark":>15} {"✓":>4}')
print('─' * 70)
for _, row in sanity_s2.iterrows():
    scenario = f"{int(row['recovery_pct'])}% recovery"
    buyers = f"{int(row['incremental_buyers']):,}"
    total = f"R${row['revenue_ltv_m']:.3f}M"
    per_buyer = f"R${row['revenue_per_buyer']:.0f}"
    bench = f"R${row['benchmark']:.0f}"
    status = '✓' if row['within_tolerance'] else '✗'
    print(f"{scenario:<15} {buyers:>15} {total:>15} {per_buyer:>15} {bench:>15} {status:>4}")
print()

In [ ]:
cohort = pd.read_sql('''
SELECT cohort_month, cohort_quality, COUNT(*) AS cohort_size,
       ROUND(AVG(CAST(r.repeat_within_90d AS REAL))*100,2) AS retention_rate_90d
FROM csm_mat c
JOIN crsm_mat r ON c.customer_unique_id = r.customer_unique_id
WHERE cohort_month >= '2017-11-01' AND cohort_month <= '2018-01-31' AND r.repeat_within_90d IS NOT NULL
GROUP BY cohort_month
''', conn)

weak = cohort  # All Nov-Jan cohorts are considered "weak" quality
weak_total = weak['cohort_size'].sum()
weak_rate = weak['retention_rate_90d'].mean() / 100

# Get strong cohort for comparison
strong_cohort = pd.read_sql('''
SELECT AVG(CAST(r.repeat_within_90d AS REAL))*100 AS retention_rate_90d
FROM csm_mat c
JOIN crsm_mat r ON c.customer_unique_id = r.customer_unique_id
WHERE cohort_month >= '2017-06-01' AND cohort_month <= '2017-09-30' AND r.repeat_within_90d IS NOT NULL
''', conn)

strong_rate = strong_cohort.iloc[0]['retention_rate_90d'] / 100
gap_pp = (strong_rate - weak_rate) * 100

print(f'\n═══════════════════════════════════════════════════════════════════')
print(f'SCENARIO 3: HOLIDAY COHORT RE-ENGAGEMENT')
print(f'═══════════════════════════════════════════════════════════════════')
print(f'Target: Nov 2017 – Jan 2018 holiday cohorts')
print(f'')
print(f'Holiday cohort total customers:          {int(weak_total):,}')
print(f'  Holiday cohort 90-day retention:       {weak_rate*100:.2f}%')
print(f'  Mid-year cohort 90-day retention:      {strong_rate*100:.2f}%')
print(f'  Retention gap:                         {gap_pp:.2f}pp')
print(f'')
print(f'REVENUE FORMULA (Scenario 3):')
print(f'  incremental_revenue = incremental_buyers × R${REV_REPEAT_BUYER_90D:.2f}')
print(f'  (using repeat buyer 90-day revenue from NB05)')
print(f'')

results_s3 = []
for recovery in [0.33, 0.50, 0.75]:
    gap_recovery_pp = gap_pp * recovery
    new_rate = weak_rate + (gap_recovery_pp / 100)
    incremental = int(weak_total * (new_rate - weak_rate))
    revenue = incremental * REV_REPEAT_BUYER_90D
    
    results_s3.append({
        'recovery_pct': recovery * 100,
        'incremental_buyers': incremental,
        'revenue_90d_m': revenue / 1_000_000
    })
    
    print(f'Recovery {int(recovery*100)}% of gap ({gap_recovery_pp:.2f}pp):')
    print(f'  New retention rate:                    {new_rate*100:.2f}%')
    print(f'  Incremental repeat buyers:             {incremental:,}')
    print(f'  Incremental revenue (90-day):          R${revenue/1_000_000:.2f}M')
    print(f'')

s3_df = pd.DataFrame(results_s3)

In [ ]:
sanity_s3 = s3_df.copy()
sanity_s3['revenue_per_buyer'] = sanity_s3['revenue_90d_m'] * 1_000_000 / sanity_s3['incremental_buyers']
sanity_s3['benchmark'] = REV_REPEAT_BUYER_90D
sanity_s3['within_tolerance'] = np.abs(sanity_s3['revenue_per_buyer'] - sanity_s3['benchmark']) < 1

print('SANITY CHECK TABLE (Scenario 3):')
print(f'{"Scenario":<15} {"Incr. Buyers":>15} {"Total Revenue":>15} {"Rev/Buyer":>15} {"Benchmark":>15} {"✓":>4}')
print('─' * 70)
for _, row in sanity_s3.iterrows():
    scenario = f"{int(row['recovery_pct'])}% recovery"
    buyers = f"{int(row['incremental_buyers']):,}"
    total = f"R${row['revenue_90d_m']:.2f}M"
    per_buyer = f"R${row['revenue_per_buyer']:.0f}"
    bench = f"R${row['benchmark']:.0f}"
    status = '✓' if row['within_tolerance'] else '✗'
    print(f"{scenario:<15} {buyers:>15} {total:>15} {per_buyer:>15} {bench:>15} {status:>4}")
print()

In [ ]:
s1_medium = s1_df[s1_df['uplift_pp'] == 1.0].iloc[0]
s2_medium = s2_df[s2_df['recovery_pct'] == 50].iloc[0]
s3_medium = s3_df[s3_df['recovery_pct'] == 50].iloc[0]

total_buyers = s1_medium['incremental_buyers'] + s2_medium['incremental_buyers'] + s3_medium['incremental_buyers']
total_revenue = s1_medium['revenue_90d_m'] + s2_medium['revenue_ltv_m'] + s3_medium['revenue_90d_m']

print(f'\n═══════════════════════════════════════════════════════════════════')
print(f'COMBINED IMPACT (MEDIUM-CASE, ALL SCENARIOS IN PARALLEL)')
print(f'═══════════════════════════════════════════════════════════════════')
print(f'')
print(f'Scenario 1 (Cross-Sell, +1.0pp):       {int(s1_medium["incremental_buyers"]):>8,} buyers → R${s1_medium["revenue_90d_m"]:>6.2f}M (90-day)')
print(f'Scenario 2 (Delivery Recovery, 50%):   {int(s2_medium["incremental_buyers"]):>8,} buyers → R${s2_medium["revenue_ltv_m"]:>6.3f}M (lifetime)')
print(f'Scenario 3 (Holiday Cohort, 50%):     {int(s3_medium["incremental_buyers"]):>8,} buyers → R${s3_medium["revenue_90d_m"]:>6.2f}M (90-day)')
print(f'')
print(f'TOTAL INCREMENTAL:')
print(f'  Repeat buyers:                       {int(total_buyers):,}')
print(f'  Revenue (blended):                   R${total_revenue:.2f}M')
print(f'  Context (current repeaters):         {int(baseline_repeat_buyers):,}')
print(f'  Uplift vs current:                   {(total_buyers/baseline_repeat_buyers*100):.1f}%')
print(f'')
print(f'⚠️  NOTE: Scenario 2 uses lifetime value; Scenarios 1 & 3 use 90-day revenue.')
print(f'    Not directly summable. See individual scenario details above.')

## Summary

### Revenue Formulas Used

All revenue calculations are grounded in validated benchmarks from **Notebook 05 (CLV & Churn Signals)**:

| Scenario | Formula | Benchmark | Source |
|----------|---------|-----------|--------|
| **Scenario 1: Cross-Sell** | `incremental_buyers × R$92.34` | Repeat buyer uplift (R$253 − R$161 over 90d) | `repeat_vs_one_time_summary` |
| **Scenario 2: Delivery Recovery** | `incremental_buyers × R$178.10` | Late-delivery avg lifetime revenue | `churn_signal_revenue` (NB05) |
| **Scenario 3: Holiday Cohort** | `incremental_buyers × R$253.10` | Repeat buyer avg 90-day revenue | `repeat_vs_one_time_summary` |

### What This Analysis Shows

- Each incremental repeat buyer added via cross-sell should generate approximately **R$92 in incremental 90-day revenue** (the uplift from converting a one-time buyer to a repeat buyer).
- Each incremental repeat buyer recovered from delayed delivery should generate approximately **R$178 in lifetime value**.
- Each incremental repeat buyer from holiday cohort re-engagement should generate approximately **R$253 in 90-day repeat buyer revenue**.

### Important Notes

1. **These are what-if estimates, not forecasts.** They show directional impact under stated assumptions, not guarantees.
2. **90-day vs lifetime revenue:** Scenarios 1 & 3 use 90-day horizons; Scenario 2 uses lifetime. Direct summation is not meaningful.
3. **All scenarios are independent.** They can run in parallel with no interference.
4. **Revenue timing:** 90-day observations use right-censored windows. Scenarios assume incremental buyers behave like observed repeat buyers in the dataset.

---

In [ ]:
conn.close()
print('\n✓ Analysis complete.')
print('\nKey takeaways:')
print(f'  • Cross-sell (S1): {int(s1_medium["incremental_buyers"]):,} buyers @ ~R${REV_REPEAT_UPLIFT_90D:.0f}/buyer = R${s1_medium["revenue_90d_m"]:.2f}M (90d)')
print(f'  • Delivery recovery (S2): {int(s2_medium["incremental_buyers"]):,} buyers @ ~R${REV_LATE_DELIVERY_LTV:.0f}/buyer = R${s2_medium["revenue_ltv_m"]:.3f}M (LTV)')
print(f'  • Holiday cohort (S3): {int(s3_medium["incremental_buyers"]):,} buyers @ ~R${REV_REPEAT_BUYER_90D:.0f}/buyer = R${s3_medium["revenue_90d_m"]:.2f}M (90d)')
print(f'\n  All revenue values are plausible given per-buyer benchmarks from NB05.')